<br>**Setup Environment**

In [1]:
import os
os.environ["TF_USE_LEGACY_KERAS"] = "1"

# Install (in Colab)
!pip install -q tensorflow tensorflow-model-optimization

<br>**Import**

In [2]:
# Import
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import tensorflow_model_optimization as tfmot

import numpy as np

<br>**Load dataset (MNIST)**

In [3]:
(train_images, train_labels), (test_images, test_labels) = keras.datasets.mnist.load_data()

print(train_images.shape)
print(test_images.shape)

(60000, 28, 28)
(10000, 28, 28)


<br>**Preprocessing**

In [4]:
# Flatten
train_images = train_images.reshape((60000, 28 * 28))
test_images = test_images.reshape((10000, 28 * 28))

# Normalize
train_images = train_images.astype("float32") / 255
test_images = test_images.astype("float32") / 255

# One-hot encoding
train_labels = keras.utils.to_categorical(train_labels)
test_labels = keras.utils.to_categorical(test_labels)

<br>**Build model**

In [5]:
# model = keras.Sequential([
#    layers.Dense(512, activation='relu', input_shape=(28*28,)),
#    layers.Dense(10, activation='softmax')
# ])

model = keras.Sequential([
    keras.Input(shape=(28*28,)),
    layers.Dense(512, activation='relu'),
    layers.Dense(10, activation='softmax')
])

<br>**Compile**

In [6]:
model.compile(
    optimizer='rmsprop',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

<br>**Train**

In [7]:
model.fit(
    train_images,
    train_labels,
    epochs=5,
    batch_size=120,
    validation_split=0.1
)

Epoch 1/5
450/450 [==============================] - 5s 10ms/step - loss: 0.2771 - accuracy: 0.9191 - val_loss: 0.1118 - val_accuracy: 0.9703
Epoch 2/5
450/450 [==============================] - 6s 12ms/step - loss: 0.1125 - accuracy: 0.9670 - val_loss: 0.0988 - val_accuracy: 0.9733
Epoch 3/5
450/450 [==============================] - 4s 9ms/step - loss: 0.0737 - accuracy: 0.9781 - val_loss: 0.0799 - val_accuracy: 0.9775
Epoch 4/5
450/450 [==============================] - 4s 10ms/step - loss: 0.0534 - accuracy: 0.9835 - val_loss: 0.0741 - val_accuracy: 0.9795
Epoch 5/5
450/450 [==============================] - 5s 12ms/step - loss: 0.0402 - accuracy: 0.9881 - val_loss: 0.0729 - val_accuracy: 0.9790


<br>**Evaluate**

In [8]:
test_loss, test_acc = model.evaluate(test_images, test_labels)
print("Test accuracy:", test_acc)

313/313 [==============================] - 3s 6ms/step - loss: 0.0679 - accuracy: 0.9795
Test accuracy: 0.9794999957084656


<br>**Apply PRUNING**

In [9]:
pruning_params = {
    "pruning_schedule": tfmot.sparsity.keras.PolynomialDecay(
        initial_sparsity=0.0,
        final_sparsity=0.5,
        begin_step=0,
        end_step=1000
    )
}

model_pruned = tfmot.sparsity.keras.prune_low_magnitude(model, **pruning_params)

<br>**Recompile pruned mode**

In [10]:
model_pruned.compile(
    optimizer='rmsprop',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

<br>**Fine-tuning**

In [12]:
callbacks = [
    tfmot.sparsity.keras.UpdatePruningStep()
]

model_pruned.fit(
    train_images,
    train_labels,
    epochs=2,
    batch_size=120,
    callbacks=callbacks
)

Epoch 1/2
500/500 [==============================] - 6s 11ms/step - loss: 0.0322 - accuracy: 0.9911
Epoch 2/2
500/500 [==============================] - 6s 12ms/step - loss: 0.0229 - accuracy: 0.9939


<br>**Strip pruning**

In [13]:
model_pruned_stripped = tfmot.sparsity.keras.strip_pruning(model_pruned)

<br>**Convert to TFLite with Dynamic Quantization**

In [14]:
converter = tf.lite.TFLiteConverter.from_keras_model(model_pruned_stripped)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
tfmodel = converter.convert()
open("digit_recognition_pruned_quant.tflite","wb").write(tfmodel)

416632

<br>**Download**

In [15]:
from google.colab import files

files.download("digit_recognition_pruned_quant.tflite")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>